## Amazon Products Data Cleaning | Feature Engineering — From Raw to Clean and Processed

Authors: 
    - JULIO DOURADO (@julio-dourado)
    - GUSTAVO RODRIGUES (@GustavoHenriqueRS)
    - LEONARDO LAGO (@lelamo2002)

License: MIT

Data: https://www.kaggle.com/datasets/ikramshah512/amazon-products-sales-dataset-42k-items-2025/data

In [1]:
import pandas as pd
import re
import os

# Carregamento dos dados
df = pd.read_csv('../bronze/data/amazon_products_sales_data_uncleaned.csv')

### ============================================================================
### SEÇÃO 2: FUNÇÕES AUXILIARES DE CONVERSÃO
### ============================================================================

## Funções para conversão de tipos e extração de dados

In [2]:
# Função para converter valores com 'k' (milhares) para inteiros
def convert_to_int(value):
    if pd.isna(value) or value == '':
        return 0
    
    value = str(value).strip()
    
    # Extrair apenas a parte numérica (incluindo ponto decimal e 'k')
    # Padrão: captura números como 300, 6k, 1.2k, 300+, etc.
    match = re.search(r'(\d+\.?\d*)[kK]?', value)
    
    if not match:
        return 0
    
    number_str = match.group(1)
    
    # Verifica se tem 'k' (milhares)
    if 'k' in value.lower():
        number = float(number_str)
        return int(number * 1000)
    else:
        return int(float(number_str))


# Função para converter preços para float
def convert_to_float(value):
    if pd.isna(value) or value == '':
        return 0.0
    
    value = str(value).strip()
    
    # Extrair apenas valores numéricos (incluindo ponto decimal)
    # Padrão: captura números como 1299.99, $1,299.99, etc.
    match = re.search(r'(\d+(?:,\d{3})*(?:\.\d+)?)', value)
    
    if not match:
        return 0.0
    
    # Remove vírgulas e converte para float
    number_str = match.group(1).replace(',', '')
    return float(number_str)


# Função para extrair percentual do cupom
def extract_coupon_percentage(coupon_text):
    if pd.isna(coupon_text) or coupon_text == '' or 'No Coupon' in str(coupon_text):
        return 0.0
    
    # Extrair número do texto "Save 10% with coupon"
    match = re.search(r'(\d+(?:\.\d+)?)%', str(coupon_text))
    if match:
        return float(match.group(1))
    return 0.0

### ============================================================================
### SEÇÃO 3: PADRONIZAÇÃO DE TIPOS (TYPE CONVERSION)
### ============================================================================

## 3.1 Conversão de Timestamps e Datas

Separando a coluna `collected_at` em `date` e `time`.

In [ ]:
df['collected_at'] = pd.to_datetime(df['collected_at'])

df['date'] = df['collected_at'].dt.normalize()
df['time'] = df['collected_at'].dt.time

## 3.2 Conversão de Numéricos

Limpando e convertendo colunas numéricas para os tipos apropriados.

In [4]:
# Rating: "4.5 out of 5 stars" -> 4.5
df['rating'] = df['rating'].str.extract(r'([\d\.]+)').astype(float)

# Reviews: remover vírgulas e converter para int
df['number_of_reviews'] = df['number_of_reviews'].str.replace(',', '').fillna('0').astype(int)

# Preço com desconto: remover $ e vírgulas, converter para float
df['current/discounted_price'] = df['current/discounted_price'].str.replace('$', '').str.replace(',', '').fillna('0').astype(float)

# Preço original: usar função auxiliar para tratar casos complexos
df['listed_price'] = df['listed_price'].apply(convert_to_float)

# Compras no último mês: usar função auxiliar para tratar 'k' e texto extra
df['bought_in_last_month'] = df['bought_in_last_month'].apply(convert_to_int)

## 3.3 Conversão para Booleanos

Transformando colunas de texto em valores True/False.

In [5]:
# Transformar is_best_seller em booleano
df['is_best_seller'] = df['is_best_seller'].apply(lambda x: True if str(x).strip() == 'Best Seller' else False)

# Transformar is_sponsored em booleano
df['is_sponsored'] = df['is_sponsored'].apply(lambda x: True if str(x).strip() == 'Sponsored' else False)

# Transformar buy_box_availability em booleano
df['buy_box_availability'] = df['buy_box_availability'].apply(lambda x: True if str(x).strip() == 'Add to cart' else False)

## 3.4 Conversão de Strings e Ajustes Finais de Tipos

In [ ]:
# Converter coluna 'title' para tipo string (mais moderno que object)
df['title'] = df['title'].astype('string')

### ============================================================================
### SEÇÃO 4: TRANSFORMAÇÃO
### ============================================================================

## 4.1 Extração de Informações de Cupom

Extraindo percentual de desconto do cupom e criando coluna booleana.

In [7]:
# Criar coluna de percentual de desconto do cupom
df['coupon_discount_pct'] = df['is_couponed'].apply(extract_coupon_percentage)

# Transformar is_couponed em booleano
df['is_couponed'] = df['coupon_discount_pct'] > 0

### ============================================================================
### SEÇÃO 5: REMOÇÃO DE COLUNAS
### ============================================================================

## Remoção de colunas não relevantes ou redundantes

In [8]:
# Lista de colunas a remover
colunas_remover = []

# Remover price_on_variant (dados inconsistentes)
if 'price_on_variant' in df.columns:
    colunas_remover.append('price_on_variant')

# Remover collected_at (já separado em date + time)
if 'collected_at' in df.columns:
    colunas_remover.append('collected_at')

# Remover delivery_details (será tratado em outra etapa)
if 'delivery_details' in df.columns:
    colunas_remover.append('delivery_details')

# Remover sustainability_badges (dados insuficientes)
if 'sustainability_badges' in df.columns:
    colunas_remover.append('sustainability_badges')

# Remover URLs (mantidos na Bronze, não necessários na Silver)
if 'image_url' in df.columns:
    colunas_remover.append('image_url')
if 'product_url' in df.columns:
    colunas_remover.append('product_url')

# Executar remoção
if colunas_remover:
    df = df.drop(columns=colunas_remover)
    print(f"✅ Colunas removidas: {', '.join(colunas_remover)}")

✅ Colunas removidas: price_on_variant, collected_at, delivery_details, sustainability_badges, image_url, product_url


### ============================================================================
### SEÇÃO 6: RENOMEAÇÃO DE COLUNAS
### ============================================================================

## Padronização de nomenclatura das colunas

In [9]:
# Renomear colunas para nomes mais descritivos
df = df.rename(columns={
    'number_of_reviews': 'total_reviews',
    'current/discounted_price': 'discounted_price',
    'listed_price': 'original_price',
    'bought_in_last_month': 'purchased_last_month',
    'is_couponed': 'has_coupon'
})

### ============================================================================
### SEÇÃO 7: VALIDAÇÃO E FILTRAGEM DE DADOS
### ============================================================================

## Filtragem de dados inválidos

Removendo registros sem informação de preço válida.

In [10]:
# Remover produtos sem preço
df = df[df['discounted_price'] > 0].copy()


### ============================================================================
### SEÇÃO 8: EXPORTAÇÃO PARA CAMADA SILVER
### ============================================================================

## Salvamento dos dados transformados e validados

In [11]:
# Criar diretório Silver se não existir
silver_dir = '../silver/data'
os.makedirs(silver_dir, exist_ok=True)

# Salvar dados transformados
output_file = f'{silver_dir}/amazon_products_cleaned.csv'
df.to_csv(output_file, index=False, encoding='utf-8')

print(f"✅ Dados salvos em: {output_file}")
print(f"📊 Shape final: {df.shape}")

display(df)

✅ Dados salvos em: ../silver/data/amazon_products_cleaned.csv
📊 Shape final: (30926, 13)


,title,rating,total_reviews,purchased_last_month,discounted_price,original_price,is_best_seller,is_sponsored,has_coupon,buy_box_availability,date,time,coupon_discount_pct
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375,300,89.68,159.00,False,True,True,True,2025-08-21,11:14:29,15.0
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457,6000,9.99,15.99,False,True,False,True,2025-08-21,11:14:29,0.0
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044,2000,314.00,349.00,False,True,False,True,2025-08-21,11:14:29,0.0
8,Complete Protect: One plan covers all eligible...,4.0,4380,0,16.99,0.00,False,False,True,False,2025-08-21,11:14:29,50.0
10,Amazon Basics 48-Pack AA Alkaline High-Perform...,4.7,865598,100000,14.99,0.00,True,False,False,True,2025-08-21,11:14:29,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
42670,"Elgato 4K Pro, Internal Capture Card: 8K60 Pas...",5.0,1,100,195.99,0.00,False,False,False,False,2025-08-30,19:56:33,0.0
42671,"Arlo Essential Spotlight Camera, Wireless Secu...",4.2,20,200,89.99,0.00,False,False,False,True,2025-08-30,19:56:33,0.0
42672,"GIGABYTE - AORUS FO32U2-32"" QD OLED Gaming Mon...",4.3,57,50,899.99,1099.99,False,False,False,True,2025-08-30,19:56:33,0.0
42673,Monoprice XLR Male to 1/4-Inch TRS Male Cable ...,4.7,7102,500,10.39,15.98,False,False,False,True,2025-08-30,19:56:33,0.0
